# Monte Carlo π — Four-run comparison

Compare simulations with **1,000**, **10,000**, **100,000**, and **1,000,000** tests.

- **Circle grid (2 × 2):** displays the points **outside** the unit circle from `results_N.csv`, with the final π estimate from `pi_N.csv` above each panel.
- **Convergence grid (2 × 2):** plots the recorded π estimates from `pi_N.csv` as the computation progressed, alongside the true value of π.

The notebook supports the updated, smaller CSV format (`count,x,y` for outside points only).

In [ ]:
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

TEST_COUNTS = [1_000, 10_000, 100_000, 1_000_000]

# Set this to the folder holding results_N.csv and pi_N.csv if needed.
# Common locations are the notebook folder, its parent folder, or ../results.
SEARCH_DIRS = [Path('.'), Path('..'), Path('../results'), Path('results')]

def find_csv(filename):
    for folder in SEARCH_DIRS:
        path = folder / filename
        if path.is_file():
            return path
    raise FileNotFoundError(
        f'Missing {filename}. Checked: ' + ', '.join(str(p / filename) for p in SEARCH_DIRS)
    )

runs = {}
for n in TEST_COUNTS:
    points_path = find_csv(f'results_{n}.csv')
    pi_path = find_csv(f'pi_{n}.csv')
    points = pd.read_csv(points_path)
    progress = pd.read_csv(pi_path).sort_values('count')
    if progress.empty:
        raise ValueError(f'{pi_path} contains no measurements')
    for column in ['x', 'y']:
        if column not in points.columns:
            raise ValueError(f'{points_path} needs a {column!r} column')
    for column in ['count', 'pi']:
        if column not in progress.columns:
            raise ValueError(f'{pi_path} needs a {column!r} column')
    runs[n] = {'points': points, 'progress': progress,
               'final_pi': float(progress.iloc[-1]['pi'])}
    print(f'{n:>10,} tests: {len(points):>9,} outside points, '
          f'{len(progress):>5,} π samples, final π = {runs[n]["final_pi"]:.8f}')

## 1. Outside-circle points — four simulations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)

for ax, n in zip(axes.flat, TEST_COUNTS):
    run = runs[n]
    points = run['points']
    ax.scatter(points['x'], points['y'], s=1.1, color='tomato',
               alpha=0.55, linewidths=0, rasterized=True)
    ax.set_title(f'{n:,} tests  |  Final π = {run["final_pi"]:.8f}', fontsize=11)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.15)

fig.suptitle('Points outside the unit circle', fontsize=15)
plt.show()

## 2. π convergence — four simulations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

for ax, n in zip(axes.flat, TEST_COUNTS):
    run = runs[n]
    progress = run['progress']
    ax.plot(progress['count'], progress['pi'], linewidth=1.4,
            label='Estimated π')
    ax.axhline(math.pi, color='tomato', linestyle='--', linewidth=1.4,
               label='Actual π')
    ax.set_title(f'{n:,} tests  |  Final π = {run["final_pi"]:.8f}', fontsize=11)
    ax.set_xlabel('Tests completed')
    ax.set_ylabel('π estimate')
    ax.ticklabel_format(axis='x', style='sci', scilimits=(4, 4))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)

fig.suptitle('Monte Carlo convergence toward π', fontsize=15)
plt.show()

## 3. Final comparison

In [ ]:
summary = pd.DataFrame([
    {'Tests': n, 'Estimated π': runs[n]['final_pi'],
     'Absolute error': abs(runs[n]['final_pi'] - math.pi),
     'Outside points saved': len(runs[n]['points'])}
    for n in TEST_COUNTS
])
display(summary.style.format({
    'Tests': '{:,.0f}', 'Estimated π': '{:.8f}',
    'Absolute error': '{:.8f}', 'Outside points saved': '{:,.0f}'
}))